# Clustering
**Jennifer Eigo - University of Connecticut - Dept. of Operations and Information Management**

-------------------------------------
In this module we will practice two different clustering techniques - hierarchical clustering and K-means clustering. Clusters help us find groups are records that are similar.



# Environment Setup

In [ ]:
# import modules

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

!pip install scikit-learn
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from pandas.plotting import parallel_coordinates
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


# Set display options to show all columns
pd.set_option('display.max_columns', None)


In [ ]:
# mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 3/ToyotaCorollaClusteringSample.csv')

In [ ]:
# shape
# show how many rows and columns
# this small sample has 150 rows and 11 columns
# we are starting with a small sample to practice
df.shape

In [ ]:
# Preview
print(df.head())

# Hierarchical Clustering - Manual Math

First we will do hierarchical clustering by hand so you can see how the math works within this technique.

Step 1 - Calculate all the pairwise distances between each row.  We will use the Euclidean distance for this.  

In [ ]:
# Select the columns for distance calculation
cols_for_distance = ['Age_08_04', 'KM', 'HP', 'CC', 'Doors', 'Weight']
df_distance = df[cols_for_distance]

# Standardize the data
# Using StandardScaler as requested
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_distance)

# Convert the scaled data back to a pandas DataFrame for easier handling (optional but good practice)
# Use the original column names
df_scaled = pd.DataFrame(df_scaled, columns=cols_for_distance)


# Calculate pairwise Euclidean distance matrix using the scaled data
n_rows = df_scaled.shape[0]
distance_matrix = np.zeros((n_rows, n_rows))

for i in range(n_rows):
    for j in range(n_rows):
        if i < j: # Calculate for the upper triangle to avoid redundant calculations and self-distance
            diff_sq = (df_scaled.iloc[i] - df_scaled.iloc[j])**2
            distance = np.sqrt(diff_sq.sum())
            distance_matrix[i, j] = distance
            distance_matrix[j, i] = distance # Fill the lower triangle as the matrix is symmetric

# The distance_matrix now contains the pairwise Euclidean distances of the scaled data.
# For a small sample, you can print or inspect it. For larger data, this matrix can be very big.
print("Pairwise Euclidean Distance Matrix (after standardization):")
print(distance_matrix)

Step 2 - Find the two records with the smallest distance.

In [ ]:
# Find the minimum distance (excluding the diagonal of zeros)
min_distance = np.min(distance_matrix[distance_matrix > 0])

# Find the indices of the first occurrence of the minimum distance
# np.where returns a tuple of arrays, one for each dimension
min_row_indices, min_col_indices = np.where(distance_matrix == min_distance)

# Since np.where finds all occurrences, and we know it's symmetric,
# we can just take the first pair of indices found.
row1_index = min_row_indices[0]
row2_index = min_col_indices[0]

print(f"\nSmallest pairwise distance found: {min_distance:.4f}")
print(f"Rows with the smallest pairwise distance (0-indexed): {row1_index} and {row2_index}")

# Print the original data for these rows
print("\nOriginal Data for Row 1:")
print(df_distance.iloc[row1_index])
print("\nOriginal Data for Row 2:")
print(df_distance.iloc[row2_index])

# Print the standardized data for these rows
print("\nStandardized Data for Row 1:")
print(df_scaled.iloc[row1_index])
print("\nStandardized Data for Row 2:")
print(df_scaled.iloc[row2_index])

Step 3 - Calculate the vector representing the center of the two combined rows.

In [ ]:
# Select the rows at indices 95 and 96 from the *scaled* DataFrame
row_95_scaled = df_scaled.iloc[95]
row_96_scaled = df_scaled.iloc[96]

# Calculate the average of these two scaled vectors
# This operation averages each corresponding element in the two rows
average_vector_scaled = (row_95_scaled + row_96_scaled) / 2

# Print the resulting average vector
print("\nAverage vector of scaled row 95 and scaled row 96:")
print(average_vector_scaled)

# The 'average_vector_scaled' is a pandas Series representing the centroid of the cluster
# if these two rows were to form a cluster using the centroid linkage method,
# calculated on the standardized data.

Step 4 - Drop the two original rows from the dataframe and add a new row for this new vector.

In [ ]:
# Convert the average_vector_scaled to a DataFrame with a single row
average_df_scaled = average_vector_scaled.to_frame().T

# Drop the original rows (95 and 96) from the scaled DataFrame
# Use .drop() with a list of indices and specify axis=0 for rows
# We create a new DataFrame `df_scaled_updated` to avoid modifying `df_scaled` in place initially
df_scaled_updated = df_scaled.drop(index=[95, 96])

# Add the new average vector row to the updated DataFrame
# Use pd.concat to combine the DataFrame without the original rows and the new row
# ignore_index=True will reset the index of the resulting DataFrame
df_scaled_updated = pd.concat([df_scaled_updated, average_df_scaled], ignore_index=True)

# Print the shape of the updated DataFrame to verify the change
print("\nShape of the scaled DataFrame after dropping original rows and adding average vector:")
print(df_scaled_updated.shape)

# Print the tail of the updated DataFrame to see the new row
print("\nTail of the scaled DataFrame with the new average vector row:")
print(df_scaled_updated.tail())

# You can verify that the new row (which will be the last row after concat)
# corresponds to the average_vector_scaled calculated in the previous step.
# For example, you can compare df_scaled_updated.iloc[-1] with average_vector_scaled.

We can see that the dataframe now has 149 records, because we dropped two original rows and added the average of the two rows.  That last row (148) represents the cluster formed by combining the two most similar cars.  

We can then repeat this logic over and over again.  At each iteration the row count in the table would drop by one as we make another join.  

Eventually we would be left with just one row and our clustering is complete!

This is already a lot of work and we didn't even do any visualizations to allow us to analyze the results.  Instead, let's see how to use libraries instead of coding from scratch.

# Hierarchical Clustering

Now let's do it the easy way! Although it may appear that there is more code in this section, we are doing both the clustering, and making visualizations to explore the results.  

In [ ]:
# Select the columns for clustering.
# Let's use the same columns as the manual calculation.
cols_for_clustering = ['Age_08_04', 'KM', 'HP', 'CC', 'Doors', 'Weight']
df_clustering = df[cols_for_clustering]

# Standardize the data.
# It's important to standardize features before hierarchical clustering
# as distance metrics are sensitive to scale.
scaler = StandardScaler()
df_scaled_clustering = scaler.fit_transform(df_clustering)

# Convert the scaled data back to a pandas DataFrame for easier handling (optional but good practice)
# Use the original column names
df_scaled_clustering = pd.DataFrame(df_scaled_clustering, columns=cols_for_clustering)

# Print the first few rows of the dataframe to verify that the data is standardized
print(df_scaled_clustering.head())

The data looks good.  Now let's do the clustering.  

In [ ]:
# Perform hierarchical clustering
# We will use the 'ward' linkage method which minimizes the variance of the clusters being merged.
# 'euclidean' is the default distance metric, which we have already used.
linked = linkage(df_scaled_clustering, 'ward')

# Plot the dendrogram
plt.figure(figsize=(10, 7))
dendrogram(linked,
           orientation='top',
           distance_sort='descending',
           show_leaf_counts=True,
           color_threshold=10) # optional color_threshold

plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.show()

The dendrogram shows the order in which the the rows were combined into clusters.  Each horizontal line indicates a join that happened.  The height of the horizontal line (as measured by the y axis) shows the distance between the two subgroups that were combined to form that join.  I hard coded the color_threshold to 10 so that the dendrogram would show five color coded clusters which matches what we choose later on in the code.  If you leave that out, it will just make a default assumption for how many colors to use.

It can be helpful to also look at the numbers so let's make a table of all the distances from which the dendrogram is formed.  

In [ ]:
# Print the distances used to generate the dendrogram
# The third column of the linkage matrix contains the distances at which merges occur.
distances = linked[:, 2]

# The linkage matrix 'linked' has shape (n-1, 4), where n is the number of original observations.
# Each row represents a merge. The third column (index 2) is the distance of the merge.
# The last column (index 3) is the number of original observations in the new cluster.

# We can sort the distances to see the order of merges
sorted_distances = np.sort(distances)
# print("\nSorted distances at which merges occurred:")
# print(sorted_distances)

# Let's create a DataFrame to show the distances and the number of clusters that would exist
# if we cut the dendrogram at that distance (approximately).
# The number of clusters starts at the number of data points (150) and decreases by 1 at each merge.
num_original_points = df_scaled_clustering.shape[0]
num_clusters_at_merge = np.arange(num_original_points - 1, 0, -1)

# Create a DataFrame with distances and corresponding number of clusters
distances_df = pd.DataFrame({
    'Merge Distance': sorted_distances,
    'Number of Clusters After Merge': num_clusters_at_merge
})

# Print the DataFrame
print("\nMerge Distances and Corresponding Number of Clusters:")
display(distances_df)

# You can examine this table and the dendrogram to help decide on a suitable number of clusters
# by looking for large jumps in the merge distance.

Note that the distances increase as the the number of clusters decrease.  That's because each subsequent join brings together groups that are a little more different than the previous join. In the beginning we are combining rows that are very similar with very small, nearly zero deistances.  By the end we are joining clusters with distances of 6 or more.  The most interesting part of this is the last several joins, assuming you don't want a huge number of clusters.

A picture of the distances can help us pick the optimal number of clusters.

In [ ]:
# Plot the distance and number of clusters
plt.figure(figsize=(10, 6))
plt.plot(distances_df['Number of Clusters After Merge'], distances_df['Merge Distance'], marker='o', linestyle='-')
plt.title('Merge Distance vs. Number of Clusters')
plt.xlabel('Number of Clusters')
plt.ylabel('Merge Distance')
plt.grid(True)
plt.gca().invert_xaxis() # Invert x-axis so 1 cluster is on the right
plt.show()

Here we have a plot of merge distance by the number of clusters, descending.  As the number of clusters decreases, we are forced to combine groups of rows that are increasingly different.  

When interpreting this type of plot, we look for an elbow in the line which shows a big jump in how different things are.  

In this plot, 5 clusters stands out to me as an important point.  See how there are several joins with a distance of 9 and then it jumps up to 11?  Admittedly, this is a bit of a judgement call that also relies on the business situation.

Now let's save the cluster numbers for each row for 5 clusters.  We will do this for both the original data and the standardized data.  

In [ ]:
# Get the cluster assignments for k=5
k = 5
clusters = fcluster(linked, t=k, criterion='maxclust')

# Assign the cluster labels to the dataframe
df['Cluster'] = clusters

# Print the first few rows of the dataframe to verify that the cluster labels have been assigned
print(df.head())

In [ ]:
# Assign the cluster labels to the standardized dataframe
df_scaled_clustering['Cluster'] = clusters

# Print the first few rows of the dataframe to verify that the standardized data has the clusters added too
print(df_scaled_clustering.head())

Let's explore the characteristics of the clusters.

In [ ]:
# Parallel Coordinates Plot
# The parallel coordinates plot is useful for visualizing the values of each variable
# for records belonging to different clusters. It helps see the profile of each cluster.
# It's best used with scaled data.

# Select the columns to plot (all numeric columns except 'Cluster')
cols_for_plotting = cols_for_clustering + ['Cluster'] # Include the Cluster column for coloring
df_plot = df_scaled_clustering[cols_for_plotting]

# To get a better sense of the cluster profiles, we can calculate the mean
# of the scaled variables for each cluster.
cluster_means_scaled = df_scaled_clustering.groupby('Cluster')[cols_for_clustering].mean()

print("\nMean of Scaled Variables per Cluster:")
print(cluster_means_scaled)

# Add cluster counts
cluster_counts = df_scaled_clustering['Cluster'].value_counts().sort_index()
print("\nNumber of Data Points per Cluster:")
print(cluster_counts)

# We can visualize these means on a parallel coordinates plot
plt.figure(figsize=(12, 8))
parallel_coordinates(cluster_means_scaled.reset_index(), 'Cluster', colormap=plt.get_cmap("viridis")) # Use reset_index() to make Cluster a column again

plt.title('Parallel Coordinates Plot of Mean Scaled Values by Cluster')
plt.ylabel('Mean Scaled Value')
plt.xticks(rotation=15)
plt.grid(True)
plt.show()

From this, we can make observatoins about the clusters.  Cluster 2 is the biggest with 51 cars and they are the cars that have the smallest engines, fewest doors, and are the lightest.  Cluster 3 occurs the least frequently with only 5 cars in this group.  These cars are the newest, have the most horse power, and are the heaviest.  

In [ ]:
# To plot individual rows for each cluster, we need to select the columns for plotting,
# including the assigned 'Cluster' column.
cols_for_plotting_individual = cols_for_clustering + ['Cluster']
df_plot_individual = df_scaled_clustering[cols_for_plotting_individual].copy()

# We need to handle the index for the parallel coordinates plot function
# parallel_coordinates expects the 'class' column to be separate
# Let's iterate through each cluster and create a separate plot

unique_clusters = df_plot_individual['Cluster'].unique()
unique_clusters.sort() # Sort cluster numbers for order

# Determine the number of rows and columns for the subplot grid
n_clusters = len(unique_clusters)
n_cols = 3
n_rows = (n_clusters + n_cols - 1) // n_cols # Calculate number of rows needed

# Create a single figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 5)) # Adjust figsize as needed
axes = axes.flatten() # Flatten the axes array for easier iteration

# Determine the global minimum and maximum values across all features and all data points for consistent y-axis
global_min = df_scaled_clustering[cols_for_clustering].values.min()
global_max = df_scaled_clustering[cols_for_clustering].values.max()

for i, cluster_id in enumerate(unique_clusters):
    # Filter the dataframe for the current cluster
    df_cluster_subset = df_plot_individual[df_plot_individual['Cluster'] == cluster_id].copy()

    # The 'class' column (Cluster) needs to be the last column for parallel_coordinates
    # and we need to ensure the data columns are in the correct order
    data_cols = cols_for_clustering # Ensure data columns are in the original order
    df_cluster_subset = df_cluster_subset[data_cols + ['Cluster']]

    # Plot parallel coordinates for the individual rows in this cluster on the current subplot
    parallel_coordinates(df_cluster_subset, 'Cluster', colormap=plt.get_cmap("viridis"), ax=axes[i])

    axes[i].set_title(f'Cluster {cluster_id}')
    axes[i].set_ylabel('Scaled Value')
    axes[i].tick_params(axis='x', rotation=15)
    axes[i].grid(True)
    axes[i].set_ylim(global_min, global_max) # Set the same y-axis limits for all subplots


# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout() # Adjust layout to prevent titles/labels overlapping
plt.suptitle('Parallel Coordinates Plots for Each Cluster (Individual Rows)', y=1.02, fontsize=16) # Add a main title
plt.show()

This gives us a row level view of how cohesive each attribute of each cluster is.  When the lines are close together, the more similar the cars are.  The more spread out they are, the more different.  

"Good" clusters have small variability within clusters and large variability between clusters.  

Cluster 4 has no variablility in CC, cluster 1 has a little, and the other cluster all have more.  Cluster 3 has a lot of variability in weight and HP.  

Now let's look at cluster separation.

In [ ]:
# Biplots using Principal Component Analysis (PCA)
# This helps visualize clusters in a reduced-dimensional space.

from sklearn.decomposition import PCA
# import matplotlib.patches as patches # Import patches for drawing ellipses - not needed for convex hull
from scipy.spatial import ConvexHull # Import ConvexHull
import matplotlib.patches as patches # Import patches for drawing polygons

# Use the scaled data used for hierarchical clustering
# This is df_scaled_clustering, excluding the 'Cluster' column which was added later
df_scaled_pca_input = df_scaled_clustering.drop(columns=['Cluster'])

# Perform PCA - let's keep 2 components for a 2D biplot
pca = PCA(n_components=2)
principal_components = pca.fit_transform(df_scaled_pca_input)

# Create a DataFrame with the principal components
df_principal_components = pd.DataFrame(data=principal_components, columns=['principal_component_1', 'principal_component_2'])

# Add the hierarchical cluster labels to the PCA dataframe
# Ensure the index aligns with the original scaled data before PCA
df_principal_components['Cluster'] = df_scaled_clustering['Cluster']

# --- Plotting the Biplot ---

plt.figure(figsize=(12, 8))
ax = plt.gca() # Get the current axes for adding patches

# Create a scatter plot of the data points in the PCA space, colored by cluster
sns.scatterplot(x='principal_component_1', y='principal_component_2', hue='Cluster', data=df_principal_components, palette='viridis', legend='full', ax=ax)

# Add Convex Hulls for each cluster
for cluster_id in df_principal_components['Cluster'].unique():
    # Filter data for the current cluster
    cluster_data = df_principal_components[df_principal_components['Cluster'] == cluster_id]

    if len(cluster_data) >= 3: # Need at least 3 points to form a polygon
        # Get the points for the convex hull
        points = cluster_data[['principal_component_1', 'principal_component_2']].values

        # Calculate the convex hull
        hull = ConvexHull(points)

        # Get the vertices of the convex hull in order
        hull_points = points[hull.vertices]

        # Add the polygon patch for the convex hull
        polygon = patches.Polygon(hull_points,
                                  color=sns.color_palette('viridis', n_colors=len(df_principal_components['Cluster'].unique()))[cluster_id - 1], # Get color based on cluster_id
                                  alpha=0.2, fill=True)

        # Add the polygon to the plot
        ax.add_patch(polygon)
    elif len(cluster_data) > 0: # Handle clusters with 1 or 2 points (cannot form a hull)
         print(f"Cluster {cluster_id} has {len(cluster_data)} points and cannot form a Convex Hull.")


plt.title('PCA Plot of Hierarchical Clusters with Convex Hulls') # Updated title
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]:.2f} Variance Explained)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]:.2f} Variance Explained)')
plt.grid(True)
plt.axhline(0, color='grey', lw=0.5)
plt.axvline(0, color='grey', lw=0.5)
plt.show()

From this we can see that we have good separation between clusters.  Only clusters 4 and 5 have some overlapping.  The rest are well defined.  

# K-means Clustering - Manual Math

Let's start with doing some clustering with basic math to show how the process works.  

Step 1 - Randomly assign each row to a cluster.  Note that we are using the previously standardized data.

In [ ]:
# Take a copy of the scaled dataset
df_scaled_copy = df_scaled_clustering.copy()

# Drop the 'Cluster' column if it exists
if 'Cluster' in df_scaled_copy.columns:
    df_scaled_copy = df_scaled_copy.drop(columns=['Cluster'])

# Add a new column called 'Cluster 1' and randomly assign 1, 2, or 3
df_scaled_copy['Cluster 1'] = np.random.choice([1, 2, 3], size=df_scaled_copy.shape[0])

# Display the head of the dataframe to show the new column
print("\nHead of the dataframe with Cluster 1:")
print(df_scaled_copy.head())

# Print the counts of each cluster
print("Counts for Cluster 1:")
print(df_scaled_copy['Cluster 1'].value_counts().sort_index())

Step 2 - Calculate the centroid of each cluster.

In [ ]:
# Calculate the mean for each cluster
cluster_centroids = df_scaled_copy.groupby('Cluster 1')[cols_for_clustering].mean()

print("\nCentroids for Cluster 1:")
cluster_centroids


Step 3 - Compare each row to the centroid of each of the clusters.

In [ ]:
# For each row in the scaled dataframe, calculate the Euclidean distance to each centroid.
# The cluster_centroids dataframe has the centroids for each cluster (indexed by cluster number).
# We want to calculate the distance from each row in df_scaled_copy to each centroid in cluster_centroids.

# Initialize a DataFrame to store distances to each centroid for each row
distance_to_centroids = pd.DataFrame(index=df_scaled_copy.index)

# Iterate through each cluster's centroid
for cluster_id, centroid in cluster_centroids.iterrows():
    # Calculate the squared difference between each row and the current centroid
    # We only compare against the feature columns (excluding 'Cluster 1' from df_scaled_copy)
    squared_diff = (df_scaled_copy[cols_for_clustering] - centroid)**2

    # Sum the squared differences across the feature columns for each row
    sum_squared_diff = squared_diff.sum(axis=1)

    # Take the square root to get the Euclidean distance
    distances = np.sqrt(sum_squared_diff)

    # Store these distances in our distance_to_centroids DataFrame, named by the cluster ID
    distance_to_centroids[f'Distance_to_Centroid_{cluster_id}'] = distances

print("\nDistances of each row to each centroid:")
print(distance_to_centroids.head())

Step 4 - Reassign each row to the cluster that it's closest to.

In [ ]:
# For each row, find the cluster ID corresponding to the minimum distance
# Use idxmin(axis=1) to find the column name (which contains the cluster ID) with the minimum value for each row
closest_centroid_column = distance_to_centroids.idxmin(axis=1)

# Extract the cluster ID number from the column name (e.g., 'Distance_to_Centroid_1' -> 1)
# We use a lambda function to split the string and take the last part, converting it to int
df_scaled_copy['Cluster 2'] = closest_centroid_column.apply(lambda x: int(x.split('_')[-1]))

# Display the head of the dataframe to show the new cluster assignments
print("\nHead of the dataframe with Cluster 2 assignments based on closest centroid:")
print(df_scaled_copy.head())

# Print the counts of each cluster in the new assignment
print("\nCounts for Cluster 2:")
print(df_scaled_copy['Cluster 2'].value_counts().sort_index())


Step 5 - Count how many cluster assignments changed.

In [ ]:
# Compare the new cluster assignments ('Cluster 2') to the original assignments ('Cluster 1').
# Count how many rows changed clusters
num_rows_changed = (df_scaled_copy['Cluster 1'] != df_scaled_copy['Cluster 2']).sum()

print(f"\nNumber of rows that changed clusters from Cluster 1 to Cluster 2: {num_rows_changed}")

# If num_rows_changed is not zero, we would repeat steps 2 through 5.
# The K-means algorithm converges when the cluster assignments no longer change
# between iterations, or when the centroids no longer change significantly.

# Print the distribution of clusters for both assignments
print("\nCluster distribution comparison:")
print("Cluster 1:")
print(df_scaled_copy['Cluster 1'].value_counts().sort_index())
print("\nCluster 2:")
print(df_scaled_copy['Cluster 2'].value_counts().sort_index())

# Show how many records changed between specific clusters
print("\nChanges in cluster assignments between Cluster 1 and Cluster 2:")
# Create a cross-tabulation to show the counts of records from each Cluster 1 group
# that were assigned to each Cluster 2 group.
cluster_changes_crosstab = pd.crosstab(df_scaled_copy['Cluster 1'], df_scaled_copy['Cluster 2'])
print(cluster_changes_crosstab)

Repeat steps 2-5 until the clusters stabilize... We can see from these results that there were still lots of changes!  

*   26 cars stayed in Cluster 1
*   11 cars moved from Cluster 1 to Cluster 2
*   16 cars moved from Cluster 1 to Cluster 3
*   16 cars moved from Cluster 2 to Cluster 1
*   14 cars stayed in Cluster 2
*   and so on...

# K-means Clustering

With the help of the Kmeans module, we can do K-means clustering with a single line of code! Of course, we will still need to set up our data and then explore the results. For this example we will use K=3.

In [ ]:
# Ensure we are working with the original scaled data used for the manual steps
# This is df_scaled, which was created after standardizing cols_for_distance
# And importantly, does NOT have the manually added 'Cluster 1' or 'Cluster 2' columns.
# If the previous cells were run in order, df_scaled contains the correctly scaled data.
# Let's re-select the data just to be safe, using the columns we intended to cluster on.
cols_for_kmeans = ['Age_08_04', 'KM', 'HP', 'CC', 'Doors', 'Weight']
df_scaled_kmeans_input = df_scaled[cols_for_kmeans].copy()

# Apply K-means clustering
# Initialize the KMeans model with k=3 (n_clusters=3)
# random_state ensures reproducibility
# n_init=10 is optional but recommended to try multiple centroid seeds
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

# Fit the model to the scaled data and predict the cluster assignments
df_scaled_kmeans_input['Cluster'] = kmeans.fit_predict(df_scaled_kmeans_input)

# Print the first few rows of the dataframe with the new cluster assignments
print("Head of the scaled dataframe with K-means cluster assignments (k=3):")
print(df_scaled_kmeans_input.head())

# Print the counts of each cluster
print("\nCounts for K-means clusters (k=3):")
print(df_scaled_kmeans_input['Cluster'].value_counts().sort_index())

# Calculate and print the final centroids
print("\nFinal Centroids for K-means clusters (k=3) on scaled data:")
print(kmeans.cluster_centers_)

# Optionally, add the K-means clusters back to the original (non-scaled) dataframe
# This requires aligning the indices correctly
df['KMeans_Cluster_k3'] = df_scaled_kmeans_input['Cluster']

print("\nHead of the original dataframe with K-means cluster assignments (k=3):")
print(df.head())

We now have every record assigned to its optimal cluster.  Let's explore the clusters we formed.

In [ ]:
# Parallel Coordinates Plot for K-means clusters
# Use the dataframe with K-means cluster assignments on scaled data (df_scaled_kmeans_input)
# The cluster assignments are in the 'Cluster' column of this DataFrame.
cols_for_plotting_kmeans = cols_for_clustering + ['Cluster'] # cols_for_clustering is already defined

# To get a better sense of the cluster profiles, calculate the mean
# of the scaled variables for each K-means cluster.
# Group by the 'Cluster' column in df_scaled_kmeans_input
cluster_means_scaled_kmeans = df_scaled_kmeans_input.groupby('Cluster')[cols_for_clustering].mean()

print("\nMean of Scaled Variables per K-means Cluster:")
print(cluster_means_scaled_kmeans)

# Add cluster counts for K-means clusters
cluster_counts_kmeans = df_scaled_kmeans_input['Cluster'].value_counts().sort_index()
print("\nNumber of Data Points per K-means Cluster:")
print(cluster_counts_kmeans)

# We can visualize these means on a parallel coordinates plot
plt.figure(figsize=(12, 8))
# Use cluster_means_scaled_kmeans and the 'Cluster' column after resetting the index
parallel_coordinates(cluster_means_scaled_kmeans.reset_index(), 'Cluster', colormap=plt.get_cmap("viridis"))

plt.title('Parallel Coordinates Plot of Mean Scaled Values by K-means Cluster')
plt.ylabel('Mean Scaled Value')
plt.xticks(rotation=15)
plt.grid(True)
plt.show()

Cluster 2 has cars that are heavy, have large engines, and low horsepower.  Cluster 0 has the cars that are the newest that have been diven the least and have the highest horsepower.

In [ ]:
# To plot individual rows for each cluster, we need to select the columns for plotting,
# including the assigned 'Cluster' column from the K-means results.
cols_for_plotting_individual_kmeans = cols_for_clustering + ['Cluster']
# Use the dataframe with K-means cluster assignments on scaled data (df_scaled_kmeans_input)
df_plot_individual_kmeans = df_scaled_kmeans_input[cols_for_plotting_individual_kmeans].copy()

# We need to handle the index for the parallel coordinates plot function
# parallel_coordinates expects the 'class' column to be separate
# Let's iterate through each K-means cluster and create a separate plot

unique_clusters_kmeans = df_plot_individual_kmeans['Cluster'].unique()
unique_clusters_kmeans.sort() # Sort cluster numbers for order

# Determine the number of rows and columns for the subplot grid
n_clusters_kmeans = len(unique_clusters_kmeans)
n_cols_kmeans = 3 # Keep 3 columns as before
n_rows_kmeans = (n_clusters_kmeans + n_cols_kmeans - 1) // n_cols_kmeans # Calculate number of rows needed

# Create a single figure with subplots
fig_kmeans, axes_kmeans = plt.subplots(n_rows_kmeans, n_cols_kmeans, figsize=(n_cols_kmeans * 6, n_rows_kmeans * 5)) # Adjust figsize as needed
axes_kmeans = axes_kmeans.flatten() # Flatten the axes array for easier iteration

# Determine the global minimum and maximum values across all features and all data points for consistent y-axis
# Use the scaled data that was input to K-means for consistent scaling
global_min_kmeans = df_scaled_kmeans_input[cols_for_clustering].values.min()
global_max_kmeans = df_scaled_kmeans_input[cols_for_clustering].values.max()

for i, cluster_id_kmeans in enumerate(unique_clusters_kmeans):
    # Filter the dataframe for the current K-means cluster
    df_cluster_subset_kmeans = df_plot_individual_kmeans[df_plot_individual_kmeans['Cluster'] == cluster_id_kmeans].copy()

    # The 'class' column (Cluster) needs to be the last column for parallel_coordinates
    # and we need to ensure the data columns are in the correct order
    data_cols_kmeans = cols_for_clustering # Ensure data columns are in the original order
    df_cluster_subset_kmeans = df_cluster_subset_kmeans[data_cols_kmeans + ['Cluster']]

    # Plot parallel coordinates for the individual rows in this K-means cluster on the current subplot
    # Use a colormap suitable for categorical data if needed, or keep viridis
    parallel_coordinates(df_cluster_subset_kmeans, 'Cluster', colormap=plt.get_cmap("viridis"), ax=axes_kmeans[i])

    axes_kmeans[i].set_title(f'K-means Cluster {cluster_id_kmeans}')
    axes_kmeans[i].set_ylabel('Scaled Value')
    axes_kmeans[i].tick_params(axis='x', rotation=15)
    axes_kmeans[i].grid(True)
    axes_kmeans[i].set_ylim(global_min_kmeans, global_max_kmeans) # Set the same y-axis limits for all subplots


# Hide any unused subplots
for j in range(i + 1, len(axes_kmeans)):
    fig_kmeans.delaxes(axes_kmeans[j])

plt.tight_layout() # Adjust layout to prevent titles/labels overlapping
plt.suptitle('Parallel Coordinates Plots for Each K-means Cluster (Individual Rows)', y=1.02, fontsize=16) # Add a main title
plt.show()

Another nice view of the row level variability within each cluster.

And let's look at our cluster separation.

In [ ]:
# Biplots using Principal Component Analysis (PCA) for K-means clusters
# This helps visualize K-means clusters in a reduced-dimensional space.

from sklearn.decomposition import PCA
import matplotlib.patches as patches # Import patches for drawing polygons
from scipy.spatial import ConvexHull # Import ConvexHull


# Use the scaled data that was input to K-means clustering
# This is df_scaled_kmeans_input, which already has the 'Cluster' column from K-means results
# We need to drop the 'Cluster' column before performing PCA
df_scaled_kmeans_pca_input = df_scaled_kmeans_input.drop(columns=['Cluster'])


# Perform PCA - let's keep 2 components for a 2D biplot
pca_kmeans = PCA(n_components=2)
principal_components_kmeans = pca_kmeans.fit_transform(df_scaled_kmeans_pca_input)

# Create a DataFrame with the principal components
df_principal_components_kmeans = pd.DataFrame(data=principal_components_kmeans, columns=['principal_component_1', 'principal_component_2'])

# Add the K-means cluster labels to the PCA dataframe
# Ensure the index aligns with the original scaled data before PCA
df_principal_components_kmeans['Cluster'] = df_scaled_kmeans_input['Cluster']


# --- Plotting the Biplot ---

plt.figure(figsize=(12, 8))
ax_kmeans = plt.gca() # Get the current axes for adding patches

# Create a scatter plot of the data points in the PCA space, colored by K-means cluster
sns.scatterplot(x='principal_component_1', y='principal_component_2', hue='Cluster', data=df_principal_components_kmeans, palette='viridis', legend='full', ax=ax_kmeans)

# Add Convex Hulls for each K-means cluster
for cluster_id_kmeans in df_principal_components_kmeans['Cluster'].unique():
    # Filter data for the current K-means cluster
    cluster_data_kmeans = df_principal_components_kmeans[df_principal_components_kmeans['Cluster'] == cluster_id_kmeans]

    if len(cluster_data_kmeans) >= 3: # Need at least 3 points to form a polygon
        # Get the points for the convex hull
        points_kmeans = cluster_data_kmeans[['principal_component_1', 'principal_component_2']].values

        # Calculate the convex hull
        hull_kmeans = ConvexHull(points_kmeans)

        # Get the vertices of the convex hull in order
        hull_points_kmeans = points_kmeans[hull_kmeans.vertices]

        # Add the polygon patch for the convex hull
        polygon_kmeans = patches.Polygon(hull_points_kmeans,
                                  color=sns.color_palette('viridis', n_colors=len(df_principal_components_kmeans['Cluster'].unique()))[cluster_id_kmeans], # Get color based on cluster_id
                                  alpha=0.2, fill=True)

        # Add the polygon to the plot
        ax_kmeans.add_patch(polygon_kmeans)
    elif len(cluster_data_kmeans) > 0: # Handle clusters with 1 or 2 points (cannot form a hull)
         print(f"K-means Cluster {cluster_id_kmeans} has {len(cluster_data_kmeans)} points and cannot form a Convex Hull.")


plt.title('PCA Plot of K-means Clusters with Convex Hulls (k=3)') # Updated title
plt.xlabel(f'Principal Component 1 ({pca_kmeans.explained_variance_ratio_[0]:.2f} Variance Explained)')
plt.ylabel(f'Principal Component 2 ({pca_kmeans.explained_variance_ratio_[1]:.2f} Variance Explained)')
plt.grid(True)
plt.axhline(0, color='grey', lw=0.5)
plt.axvline(0, color='grey', lw=0.5)
plt.show()

Looks pretty good!  Only minimal overlap.